# Oscar Training — All Experiments

This notebook runs all four experiments from Zou et al. 2017 (Table III):
1. Single-modal fALFF
2. Single-modal ReHo
3. Single-modal GM
4. Multi-modal fALFF + GM

**Designed to run on Brown's Oscar HPC cluster with a GPU node.**
Results are saved incrementally after each repeat so progress is never lost if the job is killed.

## 0. Environment Check
Verify we are on a GPU node and all dependencies are importable before starting a long run.

In [ ]:
# Confirm GPU is visible and print device info.
# On Oscar with --gres=gpu:1 this should show a CUDA device (e.g. Tesla V100 or A100).
# If it prints 'cpu', the job was not allocated a GPU — check your SLURM flags.
import sys, os
sys.path.insert(0, '..')

import torch

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print('Device: Apple MPS (running locally, not Oscar)')
else:
    device = torch.device('cpu')
    print('WARNING: no GPU found — running on CPU, will be slow')

print(f'PyTorch version: {torch.__version__}')

In [ ]:
# Verify the dataset is reachable and has the expected number of subjects.
# If this errors, the data was not transferred correctly — re-run the rsync command.
from utils.dataset import build_datasets

DATA_DIR = '../data/raw'

ds_falff = build_datasets(DATA_DIR, derivative='falff')
ds_reho  = build_datasets(DATA_DIR, derivative='reho')
ds_gm    = build_datasets(DATA_DIR, derivative='gm')
ds_multi = build_datasets(DATA_DIR, multi_modal=True)

print(f'fALFF subjects : {len(ds_falff)}')
print(f'ReHo  subjects : {len(ds_reho)}')
print(f'GM    subjects : {len(ds_gm)}')
print(f'Multi subjects : {len(ds_multi)}')
print('Data check passed.')

## 1. Training Configuration

These are the paper's exact hyperparameters (Zou et al. 2017, Section IV-B):
- 4-fold cross-validation repeated 50 times
- 100 epochs per fold
- SGD lr=1e-4, momentum=0.9
- StepLR: halve lr every 20 epochs
- batch_size=20 (paper); we use 20 on Oscar since GPU memory is not a constraint

Change `N_REPEATS` and `EPOCHS` to small values (e.g. 2 and 5) for a quick smoke-test.

In [ ]:
import subprocess, json, re, ast
from pathlib import Path

SCRIPTS_DIR = Path('../scripts')
OUTPUTS_DIR = Path('../outputs')
OUTPUTS_DIR.mkdir(exist_ok=True)

# --- Paper settings ---
N_REPEATS  = 50    # set to 2 for a quick smoke-test
N_FOLDS    = 4
EPOCHS     = 100   # set to 5 for a quick smoke-test
BATCH_SIZE = 20    # paper uses 20; safe on Oscar GPU

print(f'N_REPEATS={N_REPEATS}  N_FOLDS={N_FOLDS}  EPOCHS={EPOCHS}  BATCH_SIZE={BATCH_SIZE}')
print(f'Total folds to train: {N_REPEATS * N_FOLDS}')

## 2. Experiment Runner
A helper function that calls `train.py` as a subprocess, streams output live to this notebook,
and parses the final summary into a Python dict.

In [ ]:
def run_experiment(label, extra_args, paper_target=None):
    """
    Launch train.py for one experiment configuration.

    Parameters
    ----------
    label        : short name used for the output sub-directory (e.g. 'falff')
    extra_args   : list of CLI flags forwarded to train.py (e.g. ['--mode', 'single', ...])
    paper_target : expected accuracy from Table III, printed at the end for quick comparison
    """
    out_dir = OUTPUTS_DIR / label

    # Build the full command — shared hyperparameters come first, experiment-specific flags last.
    cmd = [
        sys.executable, str(SCRIPTS_DIR / 'train.py'),
        '--n-repeats',  str(N_REPEATS),
        '--n-folds',    str(N_FOLDS),
        '--epochs',     str(EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--output-dir', str(out_dir),
    ] + extra_args

    print(f'\n{"="*60}')
    print(f'EXPERIMENT : {label}')
    if paper_target:
        print(f'Paper target: {paper_target}%  (Table III, Zou et al. 2017)')
    print(f'Output dir : {out_dir}')
    print(f'Command    : {" ".join(cmd)}')
    print('='*60)

    # Run as subprocess so stdout streams live into the notebook output.
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=str(SCRIPTS_DIR),
    )
    lines = []
    for line in proc.stdout:
        print(line, end='', flush=True)  # real-time streaming
        lines.append(line)
    proc.wait()

    # Parse the three summary numbers printed at the end of train.py.
    summary = {'label': label, 'returncode': proc.returncode, 'paper_target': paper_target}
    for line in lines:
        m = re.search(r'Mean val accuracy\s*:\s*([\d.]+)%', line)
        if m: summary['mean_val_acc'] = float(m.group(1))
        m = re.search(r'Variance\s*:\s*([\d.]+)', line)
        if m: summary['variance'] = float(m.group(1))
        m = re.search(r'Best single run\s*:\s*([\d.]+)%', line)
        if m: summary['best_run'] = float(m.group(1))

    ours = summary.get('mean_val_acc', float('nan'))
    if paper_target:
        delta = ours - paper_target
        print(f'\n→ {label}: {ours:.2f}%  (paper: {paper_target}%  Δ={delta:+.2f}%)')
    else:
        print(f'\n→ {label}: {ours:.2f}%')
    return summary

all_results = []   # accumulates one dict per experiment
print('Runner ready.')

## 3. Experiment A — Single-modal fALFF

fALFF (fractional Amplitude of Low-Frequency Fluctuations) is a resting-state fMRI measure
of spontaneous brain activity. Input volume: 47×60×46 voxels after center-crop.

In [ ]:
# Single fMRI branch only. Paper Table III: 62.06%.
r = run_experiment(
    label='falff',
    extra_args=['--mode', 'single', '--derivative', 'falff'],
    paper_target=62.06,
)
all_results.append(r)

## 4. Experiment B — Single-modal ReHo

ReHo (Regional Homogeneity) measures local synchrony of fMRI BOLD signals.
Same CNN architecture and input shape as fALFF (47×60×46).

In [ ]:
# ReHo uses the same fMRI CNN branch as fALFF — only the input derivative differs.
# Paper Table III: 60.27%.
r = run_experiment(
    label='reho',
    extra_args=['--mode', 'single', '--derivative', 'reho'],
    paper_target=60.27,
)
all_results.append(r)

## 5. Experiment C — Single-modal GM

GM (grey matter density) is a structural MRI feature extracted via VBM.
Larger input volume (90×117×100) uses a bigger initial MaxPool (4×4×4) to reduce spatial dims.

In [ ]:
# Single sMRI branch only. Paper Table III: 65.43%.
r = run_experiment(
    label='gm',
    extra_args=['--mode', 'single', '--derivative', 'gm'],
    paper_target=65.43,
)
all_results.append(r)

## 6. Experiment D — Multi-modal fALFF + GM

The paper's best configuration. Both CNN branches run in parallel; their 512-dim feature
vectors are concatenated into a 1024-dim vector, which is fed into a shared FC classifier.

In [ ]:
# Multi-modal: fMRI branch (fALFF) + sMRI branch (GM) fused at the feature level.
# Paper Table III: 69.15% — highest reported accuracy.
r = run_experiment(
    label='falff_gm_multi',
    extra_args=[
        '--mode', 'multi',
        '--fmri-derivative', 'falff',
        '--smri-derivative', 'gm',
    ],
    paper_target=69.15,
)
all_results.append(r)

## 7. Results Summary vs. Paper (Table III)

In [ ]:
# Build a comparison table: our accuracy vs. paper's reported accuracy.
# Δ > 0 means we beat the paper; Δ < 0 means we are below it.
import pandas as pd

rows = []
for r in all_results:
    ours   = r.get('mean_val_acc', float('nan'))
    target = r.get('paper_target') or float('nan')
    rows.append({
        'Experiment':   r['label'],
        'Our acc (%)':  f"{ours:.2f}",
        'Paper (%)':    f"{target:.2f}",
        'Δ':            f"{ours - target:+.2f}",
        'Variance':     f"{r.get('variance', float('nan')):.2f}",
        'Best run (%)': f"{r.get('best_run', float('nan')):.2f}",
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Persist all results as JSON for downstream reporting.
summary_path = OUTPUTS_DIR / 'summary.json'
with open(summary_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'\nFull results saved to {summary_path}')

## 8. Accuracy Plot — Per-repeat Means

In [ ]:
# Each point is the mean val accuracy across 4 folds for one repeat.
# Dashed horizontal lines show the paper's reported targets for each experiment.
# Tight clustering around the mean indicates a stable model; wide spread suggests high variance.
import matplotlib.pyplot as plt

colors = {'falff': 'C0', 'reho': 'C3', 'gm': 'C1', 'falff_gm_multi': 'C2'}

fig, ax = plt.subplots(figsize=(12, 5))

for r in all_results:
    results_file = OUTPUTS_DIR / r['label'] / 'results.txt'
    if not results_file.exists():
        print(f'  {r["label"]}: results.txt not found, skipping plot')
        continue

    # Read the per-repeat means saved by train.py after every repeat.
    with open(results_file) as f:
        for line in f:
            if line.startswith('all_repeat_means:'):
                vals = ast.literal_eval(line.split(':', 1)[1].strip())
                c = colors.get(r['label'], None)
                ax.plot(range(1, len(vals)+1), vals, marker='.', color=c,
                        label=r['label'], linewidth=1)
                # Paper target as a dashed line in the same colour.
                if r.get('paper_target'):
                    ax.axhline(r['paper_target'], linestyle='--', color=c, alpha=0.4,
                               label=f"{r['label']} paper ({r['paper_target']}%)")

ax.set_xlabel('Repeat index')
ax.set_ylabel('Mean val accuracy (%)')
ax.set_title('Validation accuracy per repeat — Zou et al. 2017 replication')
ax.legend(loc='lower right', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()

# Save figure alongside the summary JSON.
fig_path = OUTPUTS_DIR / 'accuracy_per_repeat.png'
plt.savefig(fig_path, dpi=150)
print(f'Plot saved to {fig_path}')
plt.show()